In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import polars as pl
from utils import binning, exploration

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(-1)
pl.Config.set_fmt_str_lengths(80)

ID, TARGET, DATE, SEGMENT = "gdt", "ddefaut_ndb", "datdelhis", "segment"
BASE = Path("data/segments/mono_sans_engagement/base_mono_sans_engagement.parquet")

# Chargement optimisé depuis le fichier Parquet
base = pl.read_parquet(BASE)

# Répartition globale des segments
base.group_by(SEGMENT).agg(
    n=pl.len(), n_defaut=pl.col(TARGET).sum(),
    taux_defaut=pl.col(TARGET).mean()
).sort(SEGMENT)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


segment,n,n_defaut,taux_defaut
str,u32,f64,f64
"""SA_mono_sans_eng""",78118,449.0,0.005748


Le segment SA_mono_sans_eng comprend 78 118 observations, dont 449 défauts. Son taux de défaut est de 0,5748 %, ce qui signifie qu’environ 0,57 % des observations de ce segment sont en défaut, soit environ 6 défauts pour 1 000 observations.

In [47]:
valeurs_segments = sorted(base[SEGMENT].unique().to_list())
print("Segments disponibles :", valeurs_segments)

# Sélection du premier segment (Segment 1)
nom_seg1 = valeurs_segments[0]
df_seg1 = base.filter(pl.col(SEGMENT) == nom_seg1)

print(f"Segment analysé : {nom_seg1}")
print(f"Dimensions : {df_seg1.shape[0]} lignes, {df_seg1.shape[1]} colonnes")
print(f"Taux de défaut observé : {df_seg1[TARGET].mean():.2%}")

Segments disponibles : ['SA_mono_sans_eng']
Segment analysé : SA_mono_sans_eng
Dimensions : 78118 lignes, 173 colonnes
Taux de défaut observé : 0.57%


In [48]:
EXCLURE = {ID, TARGET, DATE, SEGMENT, "has_engagement", "SelectionProb", "SamplingWeight", "type_PM_PP", "code_marche_v2"}

VARS = [c for c, t in df_seg1.schema.items() if c not in EXCLURE and not t.is_temporal()]
VARS_NUM = [c for c in VARS if df_seg1.schema[c].is_numeric()]
VARS_QUALI = [c for c in VARS if c not in VARS_NUM]

print(f"{len(VARS)} variables analysées : {len(VARS_NUM)} numériques, {len(VARS_QUALI)} qualitatives")

161 variables analysées : 152 numériques, 9 qualitatives


# Analyse des valeurs manquantes

In [49]:
# 1. Taux de valeurs manquantes par variable
stats_manquants = df_seg1.select(
    [(pl.col(c).null_count() / pl.len()).alias(c) for c in VARS]
).transpose(include_header=True, header_name="variable", column_names=["pct_manquant"])

print("Top 10 variables avec le plus de manquants :")
print(stats_manquants.sort("pct_manquant", descending=True).head(10))

# 2. Statistiques des variables numériques (moyenne, écart-type, quartiles...)
print("\nStatistiques des variables numériques :")
df_seg1.select(VARS_NUM).describe()

Top 10 variables avec le plus de manquants :
shape: (10, 2)
┌────────────────────────────┬──────────────┐
│ variable                   ┆ pct_manquant │
│ ---                        ┆ ---          │
│ str                        ┆ f64          │
╞════════════════════════════╪══════════════╡
│ AGE_PM_EN_MOIS             ┆ 1.0          │
│ LISTE_CODACVPRO_modif      ┆ 1.0          │
│ LISTE_CODNAF2              ┆ 1.0          │
│ LISTE_CODETAJUR            ┆ 1.0          │
│ RATIO_LITIGES_vs_CA        ┆ 1.0          │
│ MVTAFF_STABILITE_2T_vs_12M ┆ 0.988748     │
│ CODITDBDF_PRO              ┆ 0.988095     │
│ CRTOC_TOK_0010             ┆ 0.988095     │
│ CODNAF2                    ┆ 0.987634     │
│ CODETAJUR                  ┆ 0.967229     │
└────────────────────────────┴──────────────┘

Statistiques des variables numériques :


statistic,nb_tiers,nb_crt_distincts,engagement,bilan,horsbilan,nb_AC_distincts,nb_agence_gestion_distincts,nb_produits_distincts,nb_credit_immo,nb_credit_immo__sans_garantie,nb_credit_immo__crelog,nb_credit_immo__hypotheque,nb_credit_immo__autre_OCM,nb_credit_immo__contrat_vie,nb_credit_immo__autre_garantie,engagement_immo,engagement_immo__sans_garantie,engagement_immo__crelog,engagement_immo__hypotheque,engagement_immo__autre_OCM,engagement_immo__contrat_vie,engagement_immo__autre_garantie,nb_credit_MLT,nb_credit_MLT__sans_garantie,nb_credit_MLT__interfimo,nb_credit_MLT__autre_OCM,nb_credit_MLT__PGE,nb_credit_MLT__autre_garantie,engagement_MLT,engagement_MLT__sans_garantie,engagement_MLT__interfimo,engagement_MLT__autre_OCM,engagement_MLT__PGE,engagement_MLT__autre_garantie,nb_DAV,nb_CB_diff,nb_credit_perso,nb_credit_flex,nb_credit_LCP,nb_credit_revolving,nb_credit_bail,nb_credit_tresorerie,nb_credit_garantie_emise,nb_credit_autre,nb_Neiertz,nb_PGE,engagement_DAV,engagement_CB_diff,engagement_credit_perso,engagement_credit_flex,engagement_credit_LCP,engagement_credit_revolving,engagement_credit_bail,engagement_credit_tresorerie,engagement_credit_garantie_emise,engagement_credit_autre,engagement_credit_neiertz,engagement_credit_PGE,solde_DAV_moyen,solde_debiteur,solde_crediteur,anciennete_jour_sain_M0,NBJ_ARR_SS,ARR_GDT,Dfo_max_Mm1_Mm3,Dfo_max_Mm4_Mm6,Dfo_max_Mm7_Mm9,Dfo_max_Mm10_Mm12,Dfo_max_Mm13_Mm18,Dfo_max_Mm19_Mm24,Dfo_max_Mm25_Mm36,Dfo_max_Mm37_Mm48,NBJ_ARR_SS_max_Mm1_Mm3,NBJ_ARR_SS_max_Mm4_Mm6,NBJ_ARR_SS_max_Mm7_Mm9,NBJ_ARR_SS_max_Mm10_Mm12,NBJ_ARR_SS_max_Mm13_Mm18,NBJ_ARR_SS_max_Mm19_Mm24,NBJ_ARR_SS_max_Mm25_Mm36,NBJ_ARR_SS_max_Mm37_Mm48,ANCIENNETE_modif,AGE_PP_EN_MOIS,AGE_PM_EN_MOIS,CODACVPRO_modif,MNT_HA102,CRTAD_IND_0038,CRTOC_IND_0164,CRTOC_SLD_MOY,CRTOP_AG_IND_0038,CRTAD_AG_SOLDE_T,CRTAD_AG_SOLDE_T1,CRTAD_AG_SOLDE_T2,CRTAD_AG_SOLDE_T3,CRTAD_AG_SOLDE_T4,MINSOLDE_PAR,CRTAD_AG_SOLDE_IND_0009,CRTAD_IND_0015,CRTAD_AG_NBJDE_BA,CRTAD_AG_NBJDE_BB,CRTAD_AG_NBJDE_BC,CRTOC_IND_0301,CRTOC_AG_NBJDEB_IND_0015,CRTOC_AG_LIGDEB_IND_0015,CRTOC_AG_LIGCRE_IND_0009,CRTOC_AG_NBJDEB_2T_IND_0015,CRTOC_AG_LIGDEB_2T_IND_0015,CRTOC_AG_NBJCRE_2T_IND_0009,CRTOC_AG_LIGCRE_2T_IND_0009,TOTAL_LITIGES_PRO_IND_0209,CRTAD_IND_0062,CRTAD_IND_0042,CRTAD_AG_NBECR_A,CRTAD_AG_NBECR_B,NBECR_PAR,CRTAD_AG_MVTAFF_IND_0010,CRTOC_IND_0010,CRTOC_MOY_MVTAFF_IND_0010,CRTOC_AG_MVTAFF_IND_0010,CRTOC_AG_MVTAFF_2T_IND_0010,RATIO1_PRO,RATIO3_PRO,ENCOURS_PAR,CRTAE_AG_SLDEPA,MNTAUTCRT_AG_EPTE_PAR,CRTAD_IND_0205,CRTAV_IND_0165,CRTAT_IND_0205,CRTAR_IND_0036,CRTPI_IND_0036,TOTAL_TITRES_PRO_IND_0205,TOTAL_EPS_PRO_IND_0208,NB_CCT_PRO,TX_UTIL_CCT_PRO,SOLDE_EVOLUTION_T_vs_T1,SOLDE_EVOLUTION_T_vs_T4,SOLDE_RATIO_T_vs_HIST,SOLDE_PENTE_5T,SOLDE_PENTE_4T,SOLDE_PENTE_3T,NBJDE_RATIO_A_vs_A1,RATIO_SOLDE_MOYEN_SUR_EPARGNE,RATIO_NBJDEB_NONAUTO_SUR_NBJDEB,RATIO_MOUV_AFF_SUR_EPARGNE,SOLDE_VOLATILITE_5T,SOLDE_VOLATILITE_3T,NBECR_EVOLUTION_A_vs_A1,MVTAFF_STABILITE_2T_vs_12M,PART_EPARGNE_DANS_AVOIRS,RATIO_LITIGES_vs_CA,RATIO_LITIGES_vs_AVOIRS,RATIO_ECR,cod_axe_unite
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",78118.0,78118.0,78118.0,78118.0,78118.0,78118.0,78118.0,78118.0,78118.0,78118.0,78118.0,78118.0,78118.0,78118.0,78118.0,78118.0,78118.0,78118.0,78118.0,78118.0,78118.0,78118.0,78118.0,78118.0,78118.0,78118.0,78118.0,78118.0,78118.0,78118.0,78118.0,78118.0,78118.0,78118.0,78118.0,78118.0,78118.0,78118.0,78118.0,78118.0

L'analyse des valeurs manquantes met en évidence une forte concentration de données absentes sur certaines variables. Parmi les dix variables présentant les taux de valeurs manquantes les plus élevés, cinq sont entièrement manquantes : `AGE_PM_EN_MOIS`, `LISTE_CODACVPRO_modif`, `LISTE_CODNAF2`, `LISTE_CODETAJUR` et `RATIO_LITIGES_vs_CA`, avec un taux de valeurs manquantes de 100 %. Ces variables ne contiennent donc aucune information exploitable directement dans l'échantillon étudié.

Les cinq autres variables présentent également des taux de valeurs manquantes très élevés, compris entre 98,76 % et 98,87 %. En particulier, `MVTAFF_STABILITE_2T_vs_12M`, `CODITDBDF_PRO`, `CRTOC_TOK_0010`, `flux_pro` et `CODNAF2` ne disposent que d'une faible proportion de valeurs renseignées.

**Implications pour la modélisation :**

* **Exclusion potentielle :** les variables entièrement manquantes doivent être exclues du modèle en l'état, puisqu'elles ne permettent ni de distinguer les observations ni d'estimer une relation avec la variable cible. Leur définition et leur disponibilité doivent néanmoins être vérifiées avant une suppression définitive.
* **Faible apport informationnel potentiel :** les variables présentant près de 99 % de valeurs manquantes risquent d'apporter peu d'information exploitable et de rendre les estimations instables si elles sont conservées sans traitement adapté.
* **Recherche des causes :** il convient de vérifier si ces absences résultent d'un problème d'extraction, d'une indisponibilité des données ou de caractéristiques structurelles du portefeuille. Une variable absente uniquement pour certains segments de clientèle peut, par exemple, refléter une différence de couverture des données.
* **Prévention des biais :** une imputation systématique n'est pas recommandée à ce stade. Elle pourrait introduire une information artificielle et masquer les problèmes de qualité des données.

En conclusion, ces résultats justifient un premier filtrage des variables selon leur taux de valeurs manquantes, suivi d'une analyse métier des variables les plus incomplètes. Le seuil d'exclusion devra être défini en fonction de la qualité des données, de leur pertinence pour le risque de crédit et de leur disponibilité en conditions réelles d'utilisation du modèle. Les décisions de traitement devront être prises sur l'échantillon d'entraînement, puis appliquées sans réestimation aux échantillons de validation et de test.


## 3.2 Variables constantes ou presque


In [51]:
# Rapport des variables constantes et quasi constantes
const = exploration.constant_report(base, VARS, seuil=0.99)

# Nombre de variables quasi constantes
n_quasi = const.filter(
    pl.col("quasi_constante")
).height

print("Nombre de variables quasi constantes :", n_quasi)

# Afficher les variables quasi constantes
print(
    const.filter(pl.col("quasi_constante"))
         .sort("part_top", descending=True)
)

Nombre de variables quasi constantes : 79
shape: (79, 5)
┌─────────────────────────────────┬───────────┬────────────┬──────────┬─────────────────┐
│ variable                        ┆ n_valeurs ┆ valeur_top ┆ part_top ┆ quasi_constante │
│ ---                             ┆ ---       ┆ ---        ┆ ---      ┆ ---             │
│ str                             ┆ i64       ┆ str        ┆ f64      ┆ bool            │
╞═════════════════════════════════╪═══════════╪════════════╪══════════╪═════════════════╡
│ nb_tiers                        ┆ 1         ┆ 1.0        ┆ 1.0      ┆ true            │
│ engagement                      ┆ 1         ┆ 0.0        ┆ 1.0      ┆ true            │
│ bilan                           ┆ 1         ┆ 0.0        ┆ 1.0      ┆ true            │
│ horsbilan                       ┆ 1         ┆ 0.0        ┆ 1.0      ┆ true            │
│ nb_credit_immo                  ┆ 1         ┆ 0.0        ┆ 1.0      ┆ true            │
│ nb_credit_immo__sans_garantie   ┆ 1      

In [52]:

# Variables strictement constantes
constantes_strictes = const.filter(
    pl.col("n_valeurs") == 1
)

# Variables quasi constantes mais non constantes
quasi_constantes = const.filter(
    (pl.col("part_top") >= 0.99)
    & (pl.col("n_valeurs") > 1)
)

print("Variables strictement constantes :", constantes_strictes.height)
print("Variables quasi constantes :", quasi_constantes.height)

print(
    quasi_constantes.select(
        "variable", "n_valeurs", "valeur_top", "part_top"
    ).sort("part_top", descending=True)
)

Variables strictement constantes : 64
Variables quasi constantes : 15
shape: (15, 4)
┌─────────────────────────────┬───────────┬────────────┬──────────┐
│ variable                    ┆ n_valeurs ┆ valeur_top ┆ part_top │
│ ---                         ┆ ---       ┆ ---        ┆ ---      │
│ str                         ┆ i64       ┆ str        ┆ f64      │
╞═════════════════════════════╪═══════════╪════════════╪══════════╡
│ CRTAD_IND_0062              ┆ 31        ┆ 0.0        ┆ 0.999616 │
│ TX_UTIL_CCT_PRO             ┆ 45        ┆ 0.0        ┆ 0.999437 │
│ CRTOC_IND_0301              ┆ 40        ┆ 0.0        ┆ 0.999258 │
│ NB_CCT_PRO                  ┆ 2         ┆ 0.0        ┆ 0.998067 │
│ CRTOC_AG_NBJDEB_IND_0015    ┆ 59        ┆ 0.0        ┆ 0.997862 │
│ CRTOC_AG_NBJDEB_2T_IND_0015 ┆ 89        ┆ 0.0        ┆ 0.997299 │
│ CRTOC_AG_LIGDEB_2T_IND_0015 ┆ 205       ┆ 0.0        ┆ 0.997299 │
│ RATIO3_PRO                  ┆ 125       ┆ 0.0        ┆ 0.996825 │
│ MNTAUTCRT_AG_EPTE_PAR       ┆

Après exclusion des variables strictement constantes, 15 variables présentent une proportion de leur modalité dominante supérieure ou égale à 99 %. La modalité la plus fréquente est zéro pour 14 d'entre elles, tandis que `CODITDBDF_PAR` est dominée par la valeur `"0000"`. Cette concentration traduit une faible variabilité des variables sur l'échantillon d'entraînement.

Les cas les plus marqués sont `CRTAD_IND_0062`, `TX_UTIL_CCT_PRO` et `CRTOC_IND_0301`, dont la modalité dominante représente respectivement 99,96 %, 99,94 % et 99,93 % des observations. D'autres variables, comme `CRTOC_IND_0010` et `CRTOC_MOY_MVTAFF_IND_0010`, présentent une diversité importante de valeurs distinctes, mais restent dominées à 99,01 % par la valeur zéro. Le nombre de modalités ne suffit donc pas, à lui seul, à caractériser leur variabilité effective.

**Implications pour la modélisation :**

* **Pouvoir discriminant potentiellement limité :** la faible fréquence des modalités alternatives peut réduire la capacité de ces variables à distinguer les emprunteurs selon leur risque de défaut.
* **Risque d'estimations instables :** dans une régression logistique, les modalités rares peuvent entraîner des coefficients peu précis, voire des problèmes de séparation lorsque les événements de défaut sont très peu nombreux.
* **Vérification de la pertinence métier :** certaines variables peuvent néanmoins capturer des situations spécifiques, comme un incident de paiement ou une utilisation particulière du crédit. Leur faible fréquence ne justifie donc pas automatiquement leur suppression.
* **Sélection raisonnée :** la décision de conserver ou d'exclure ces variables doit tenir compte du nombre d'observations et de défauts associés aux modalités rares, de leur association avec la cible et de leur stabilité sur les échantillons de validation et de test.

En conclusion, ces 15 variables sont candidates à une sélection complémentaire, mais ne doivent pas être supprimées automatiquement sur la seule base du seuil de 99 %. Une analyse de leur distribution conditionnelle à la cible `ddefaut_ndb` permettra de mieux évaluer leur intérêt pour la prédiction du défaut.


## 3.6 Pouvoir discriminant (IV)
IV calculée sur un découpage fin (10 quantiles, manquant en classe à part), avec les seuils de Siddiqi : < 0,02 inutile, 0,02-0,1 faible, 0,1-0,3 moyen, 0,3-0,5 fort, > 0,5 suspect. Une IV au-dessus de 0,5 n'est pas forcément un problème (les variables de compte sont très prédictives en Retail), mais il faut vérifier que la variable est bien connue avant la date d'observation.

In [53]:
ivs = binning.iv_ranking(base, VARS, TARGET)
# Répartition des variables par pouvoir prédictif
print(
    ivs.group_by("pouvoir")
       .len()
       .sort("len", descending=True)
)
# Top 30 des variables selon leur IV décroissante
print(
    ivs.sort("iv", descending=True)
       .head(30)
)

shape: (5, 2)
┌─────────┬─────┐
│ pouvoir ┆ len │
│ ---     ┆ --- │
│ str     ┆ u32 │
╞═════════╪═════╡
│ inutile ┆ 80  │
│ faible  ┆ 25  │
│ moyen   ┆ 23  │
│ suspect ┆ 19  │
│ fort    ┆ 14  │
└─────────┴─────┘
shape: (30, 5)
┌─────────────────────────────────┬──────────┬─────────┬───────────┬──────────────┐
│ variable                        ┆ iv       ┆ pouvoir ┆ n_classes ┆ min_n_classe │
│ ---                             ┆ ---      ┆ ---     ┆ ---       ┆ ---          │
│ str                             ┆ f64      ┆ str     ┆ i64       ┆ i64          │
╞═════════════════════════════════╪══════════╪═════════╪═══════════╪══════════════╡
│ MINSOLDE_PAR                    ┆ 0.805318 ┆ suspect ┆ 9         ┆ 5944         │
│ anciennete_jour_sain_M0         ┆ 0.762078 ┆ suspect ┆ 10        ┆ 13           │
│ CRTAD_AG_NBJDE_BA               ┆ 0.747067 ┆ suspect ┆ 5         ┆ 4154         │
│ solde_DAV_moyen                 ┆ 0.725596 ┆ suspect ┆ 10        ┆ 2423         │
│ CRTAD_AG_SOLDE_


L'analyse de l'Information Value (IV) met en évidence une hétérogénéité du pouvoir discriminant des variables explicatives. Sur les 175 variables analysées, 87 sont classées comme inutiles, 27 présentent un pouvoir discriminant faible, 26 un pouvoir moyen, 16 un pouvoir fort et 19 sont classées comme suspectes.

Les variables jugées inutiles représentent près de la moitié des variables étudiées (49,7 %). Elles apportent peu d'information pour distinguer les emprunteurs défaillants des emprunteurs non défaillants. À l'inverse, 69 variables sont classées dans les catégories faible, moyen ou fort, soit 39,4 % du total. Les 19 variables suspectes représentent 10,9 % des variables analysées et nécessitent une vérification particulière.

**Implications pour la modélisation :**

* **Réduction de la dimension :** les variables classées comme inutiles constituent des candidates à l'exclusion, sous réserve de vérifier les seuils utilisés et leur éventuel intérêt métier.
* **Présélection des variables :** les variables présentant un IV moyen ou fort peuvent être examinées en priorité pour construire le modèle, sans être retenues automatiquement.
* **Vigilance sur les variables suspectes :** un IV anormalement élevé peut révéler une fuite de cible, une variable construite à partir d'informations postérieures au défaut ou une séparation excessive liée à des modalités rares. Ces variables doivent être auditées avant toute utilisation.
* **Validation de la stabilité :** le pouvoir discriminant observé sur l'échantillon d'entraînement doit être confirmé sur la validation et, idéalement, sur un échantillon temporel hors période d'estimation.

En conclusion, l'analyse IV constitue une première étape de sélection univariée. Elle doit être complétée par l'analyse des valeurs manquantes, des variables constantes, de la redondance entre variables et de la stabilité temporelle afin de construire un modèle de scoring robuste.


In [54]:
VARS_NUM_CORR = [
    c for c in VARS
    if base.schema[c].is_numeric()
]

print("Variables candidates :", len(VARS))
print("Variables numériques :", len(VARS_NUM_CORR))

Variables candidates : 161
Variables numériques : 152


## Analyse des paires redondantes

In [55]:
paires = exploration.paires_redondantes(
    base,
    columns=VARS_NUM_CORR,
    seuil=0.8,
    n_max=50000,
    seed=0
)

print("Nombre de paires redondantes :", paires.height)
print(paires)

Nombre de paires redondantes : 126
shape: (126, 4)
┌─────────────────────────────┬─────────────────────────────────┬───────────┬──────────┐
│ var1                        ┆ var2                            ┆ spearman  ┆ n_commun │
│ ---                         ┆ ---                             ┆ ---       ┆ ---      │
│ str                         ┆ str                             ┆ f64       ┆ i64      │
╞═════════════════════════════╪═════════════════════════════════╪═══════════╪══════════╡
│ CRTAD_IND_0205              ┆ CRTAT_IND_0205                  ┆ 1.0       ┆ 50000    │
│ CRTAD_IND_0062              ┆ RATIO_LITIGES_vs_AVOIRS         ┆ 1.0       ┆ 30062    │
│ CRTOC_IND_0010              ┆ CRTOC_MOY_MVTAFF_IND_0010       ┆ 1.0       ┆ 50000    │
│ CRTOC_AG_NBJDEB_2T_IND_0015 ┆ CRTOC_AG_LIGDEB_2T_IND_0015     ┆ -0.999999 ┆ 50000    │
│ CRTOC_IND_0301              ┆ RATIO_NBJDEB_NONAUTO_SUR_NBJDEB ┆ 0.999999  ┆ 5401     │
│ CRTOC_AG_MVTAFF_IND_0010    ┆ RATIO_MOUV_AFF_SUR_EPARGNE 

Certaines variables présentent des redondances entre elles. Lors de la phase de modélisation, nous examinerons ces redondances afin de sélectionner les variables les plus pertinentes pour notre étude, en tenant compte notamment de leur disponibilité et de leur intérêt pour la prédiction du risque de défaut.


In [56]:
paires = exploration.paires_redondantes(
    base,
    columns=VARS_NUM_CORR,
    seuil=0.8,
    n_max=50000,
    seed=0
)

print("Nombre de paires redondantes :", paires.height)
print(paires)

Nombre de paires redondantes : 126
shape: (126, 4)
┌─────────────────────────────┬─────────────────────────────────┬───────────┬──────────┐
│ var1                        ┆ var2                            ┆ spearman  ┆ n_commun │
│ ---                         ┆ ---                             ┆ ---       ┆ ---      │
│ str                         ┆ str                             ┆ f64       ┆ i64      │
╞═════════════════════════════╪═════════════════════════════════╪═══════════╪══════════╡
│ CRTAD_IND_0205              ┆ CRTAT_IND_0205                  ┆ 1.0       ┆ 50000    │
│ CRTAD_IND_0062              ┆ RATIO_LITIGES_vs_AVOIRS         ┆ 1.0       ┆ 30062    │
│ CRTOC_IND_0010              ┆ CRTOC_MOY_MVTAFF_IND_0010       ┆ 1.0       ┆ 50000    │
│ CRTOC_AG_NBJDEB_2T_IND_0015 ┆ CRTOC_AG_LIGDEB_2T_IND_0015     ┆ -0.999999 ┆ 50000    │
│ CRTOC_IND_0301              ┆ RATIO_NBJDEB_NONAUTO_SUR_NBJDEB ┆ 0.999999  ┆ 5401     │
│ CRTOC_AG_MVTAFF_IND_0010    ┆ RATIO_MOUV_AFF_SUR_EPARGNE 

## Presentation des variables avec un IV superieur à 0.5

In [27]:

import polars as pl

print(ivs.filter(pl.col("iv") > 0.5).sort(
    "iv", descending=True
))

shape: (19, 5)
┌─────────────────────────┬──────────┬─────────┬───────────┬──────────────┐
│ variable                ┆ iv       ┆ pouvoir ┆ n_classes ┆ min_n_classe │
│ ---                     ┆ ---      ┆ ---     ┆ ---       ┆ ---          │
│ str                     ┆ f64      ┆ str     ┆ i64       ┆ i64          │
╞═════════════════════════╪══════════╪═════════╪═══════════╪══════════════╡
│ MINSOLDE_PAR            ┆ 0.805318 ┆ suspect ┆ 9         ┆ 5944         │
│ anciennete_jour_sain_M0 ┆ 0.762078 ┆ suspect ┆ 10        ┆ 13           │
│ CRTAD_AG_NBJDE_BA       ┆ 0.747067 ┆ suspect ┆ 5         ┆ 4154         │
│ solde_DAV_moyen         ┆ 0.725596 ┆ suspect ┆ 10        ┆ 2423         │
│ CRTAD_AG_SOLDE_T4       ┆ 0.686286 ┆ suspect ┆ 10        ┆ 305          │
│ solde_crediteur         ┆ 0.676437 ┆ suspect ┆ 10        ┆ 2797         │
│ CODNAF2                 ┆ 0.664191 ┆ suspect ┆ 155       ┆ 1            │
│ CRTAD_AG_SOLDE_T        ┆ 0.595    ┆ suspect ┆ 10        ┆ 4882        

In [57]:

import polars as pl

print(ivs.filter(pl.col("iv") < 0.02).sort(
    "iv", descending=True
))

shape: (80, 5)
┌──────────────────────────────────┬──────────┬─────────┬───────────┬──────────────┐
│ variable                         ┆ iv       ┆ pouvoir ┆ n_classes ┆ min_n_classe │
│ ---                              ┆ ---      ┆ ---     ┆ ---       ┆ ---          │
│ str                              ┆ f64      ┆ str     ┆ i64       ┆ i64          │
╞══════════════════════════════════╪══════════╪═════════╪═══════════╪══════════════╡
│ CRTOC_SLD_MOY                    ┆ 0.019728 ┆ inutile ┆ 2         ┆ 859          │
│ CRTOC_IND_0010                   ┆ 0.019023 ┆ inutile ┆ 2         ┆ 774          │
│ CRTOC_MOY_MVTAFF_IND_0010        ┆ 0.019023 ┆ inutile ┆ 2         ┆ 774          │
│ nb_AC_distincts                  ┆ 0.016622 ┆ inutile ┆ 7         ┆ 4            │
│ CRTPI_IND_0036                   ┆ 0.012401 ┆ inutile ┆ 2         ┆ 5140         │
│ CRTOC_IND_0164                   ┆ 0.007849 ┆ inutile ┆ 2         ┆ 810          │
│ TX_UTIL_CCT_PRO                  ┆ 0.004959 ┆ in

## Variables redondantes

In [58]:
print(paires.sort("spearman", descending=True))

shape: (126, 4)
┌─────────────────────────────┬─────────────────────────────────┬───────────┬──────────┐
│ var1                        ┆ var2                            ┆ spearman  ┆ n_commun │
│ ---                         ┆ ---                             ┆ ---       ┆ ---      │
│ str                         ┆ str                             ┆ f64       ┆ i64      │
╞═════════════════════════════╪═════════════════════════════════╪═══════════╪══════════╡
│ CRTAD_IND_0205              ┆ CRTAT_IND_0205                  ┆ 1.0       ┆ 50000    │
│ CRTAD_IND_0062              ┆ RATIO_LITIGES_vs_AVOIRS         ┆ 1.0       ┆ 30062    │
│ CRTOC_IND_0010              ┆ CRTOC_MOY_MVTAFF_IND_0010       ┆ 1.0       ┆ 50000    │
│ CRTOC_IND_0301              ┆ RATIO_NBJDEB_NONAUTO_SUR_NBJDEB ┆ 0.999999  ┆ 5401     │
│ CRTOC_AG_MVTAFF_IND_0010    ┆ RATIO_MOUV_AFF_SUR_EPARGNE      ┆ 0.999971  ┆ 30062    │
│ nb_AC_distincts             ┆ nb_DAV                          ┆ 0.999855  ┆ 50000    │
│ CRT

## Analyse des correlations

### Variables quantitative 

In [60]:
import polars as pl
import pandas as pd
import numpy as np

target = "ddefaut_ndb"

# Sélectionner les variables numériques, hors cible
variables_quanti = [
    col
    for col, dtype in zip(base.columns, base.dtypes)
    if dtype.is_numeric() and col != target
]

# Conversion en pandas pour calculer les corrélations
X_quanti = base.select(variables_quanti).to_pandas()

print("Nombre de variables quantitatives :", len(variables_quanti))
print("Dimensions :", X_quanti.shape)

Nombre de variables quantitatives : 155
Dimensions : (78118, 155)


In [61]:
# 2. Calculer les matrices de corrélation
corr_pearson = X_quanti.corr(method="pearson")
corr_spearman = X_quanti.corr(method="spearman")
corr_kendall = X_quanti.corr(method="kendall")

# 3. Extraire les paires fortement corrélées
def extraire_paires(matrice, seuil=0.80):
    masque = np.triu(
        np.ones(matrice.shape, dtype=bool),
        k=1
    )

    paires = (
        matrice.where(masque)
        .stack()
        .rename("correlation")
        .reset_index()
    )

    paires.columns = ["variable_1", "variable_2", "correlation"]
    paires["correlation_absolue"] = paires["correlation"].abs()

    return (
        paires[paires["correlation_absolue"] >= seuil]
        .sort_values("correlation_absolue", ascending=False)
        .reset_index(drop=True)
    )

paires_pearson = extraire_paires(corr_pearson)
paires_spearman = extraire_paires(corr_spearman)
paires_kendall = extraire_paires(corr_kendall)

# 4. Afficher les résultats
print("\n--- Corrélations fortes : Pearson ---")
print(paires_pearson)

print("\n--- Corrélations fortes : Spearman ---")
print(paires_spearman)

print("\n--- Corrélations fortes : Kendall ---")
print(paires_kendall)


--- Corrélations fortes : Pearson ---


,variable_1,variable_2,correlation,correlation_absolue
0,CRTAD_IND_0205,CRTAT_IND_0205,1.000000,1.000000
1,SelectionProb,SamplingWeight,-1.000000,1.000000
2,nb_AC_distincts,nb_DAV,0.999926,0.999926
3,CRTAD_IND_0062,RATIO_LITIGES_vs_AVOIRS,0.999300,0.999300
4,CRTOC_SLD_MOY,CRTOC_AG_LIGCRE_2T_IND_0009,0.999266,0.999266
...,...,...,...,...
59,CRTAD_IND_0038,MINSOLDE_PAR,0.812639,0.812639
60,CRTAD_AG_SOLDE_T2,CRTAD_AG_SOLDE_IND_0009,0.808666,0.808666
61,CRTAD_IND_0042,CRTAD_AG_NBECR_B,0.807740,0.807740
62,Dfo_max_Mm4_Mm6,Dfo_max_Mm7_Mm9,0.803048,0.803048



--- Corrélations fortes : Spearman ---


,variable_1,variable_2,correlation,correlation_absolue
0,NBECR_EVOLUTION_A_vs_A1,RATIO_ECR,-1.000000,1.000000
1,SelectionProb,SamplingWeight,-1.000000,1.000000
2,CRTAD_IND_0205,CRTAT_IND_0205,1.000000,1.000000
3,CRTAD_IND_0062,RATIO_LITIGES_vs_AVOIRS,1.000000,1.000000
4,CRTOC_IND_0010,CRTOC_MOY_MVTAFF_IND_0010,1.000000,1.000000
...,...,...,...,...
122,CRTOC_IND_0164,CRTOC_IND_0010,0.808447,0.808447
123,CRTOC_IND_0164,CRTOC_MOY_MVTAFF_IND_0010,0.808440,0.808440
124,CRTAD_AG_SOLDE_T2,SOLDE_VOLATILITE_5T,0.803391,0.803391
125,Dfo_max_Mm4_Mm6,Dfo_max_Mm7_Mm9,0.803048,0.803048



--- Corrélations fortes : Kendall ---


,variable_1,variable_2,correlation,correlation_absolue
0,SelectionProb,SamplingWeight,-1.000000,1.000000
1,CRTAD_IND_0205,CRTAT_IND_0205,1.000000,1.000000
2,NBECR_EVOLUTION_A_vs_A1,RATIO_ECR,-1.000000,1.000000
3,CRTAD_IND_0062,RATIO_LITIGES_vs_AVOIRS,0.999963,0.999963
4,CRTOC_IND_0010,CRTOC_MOY_MVTAFF_IND_0010,0.999943,0.999943
...,...,...,...,...
83,CRTAD_AG_SOLDE_T1,MINSOLDE_PAR,0.810779,0.810779
84,CRTOC_IND_0164,CRTOC_IND_0010,0.805716,0.805716
85,CRTOC_IND_0164,CRTOC_MOY_MVTAFF_IND_0010,0.805697,0.805697
86,Dfo_max_Mm4_Mm6,Dfo_max_Mm7_Mm9,0.803048,0.803048


### VIF

In [62]:
import polars as pl
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

target = "ddefaut_ndb"

# 1. Sélectionner les variables quantitatives, hors cible
variables_quanti = [
    col
    for col, dtype in zip(base.columns, base.dtypes)
    if dtype.is_numeric() and col != target
]

# 2. Conversion en Pandas
X = base.select(variables_quanti).to_pandas()

# 3. Nettoyer les valeurs non finies
X = X.replace([np.inf, -np.inf], np.nan)

# 4. Éliminer les variables constantes
X = X.loc[:, X.nunique(dropna=True) > 1]

# 5. Conserver les lignes complètes pour les variables retenues
X = X.dropna()

# 6. Ajouter la constante
X_const = sm.add_constant(X, has_constant="add")

# 7. Calculer le VIF
resultats_vif = pd.baseFrame({
    "variable": X.columns,
    "VIF": [
        variance_inflation_factor(X_const.values, i + 1)
        for i in range(len(X.columns))
    ]
})

# 8. Trier par VIF décroissant
resultats_vif = resultats_vif.sort_values(
    "VIF", ascending=False
).reset_index(drop=True)

print(resultats_vif)

AttributeError: module 'pandas' has no attribute 'baseFrame'

## Variable qualitative

In [ ]:
import polars as pl
import pandas as pd
import numpy as np

from itertools import combinations
from scipy.stats import chi2_contingency

target = "ddefaut_ndb"

# 1. Identifier les variables qualitatives, hors cible
variables_quali = [
    col
    for col, dtype in zip(base.columns, base.dtypes)
    if (
        dtype in (pl.String, pl.Categorical, pl.Enum, pl.Boolean)
    )
    and col != target
]

# 2. Conversion en Pandas
X_quali = base.select(variables_quali).to_pandas()


def calculer_v_cramer(tableau):
    """Calcule le V de Cramér à partir d'un tableau de contingence."""
    chi2, p_value, ddl, effectifs_attendus = chi2_contingency(
        tableau, correction=False
    )

    n = tableau.to_numpy().sum()
    lignes, colonnes = tableau.shape

    if n == 0 or min(lignes - 1, colonnes - 1) == 0:
        return np.nan, chi2, p_value, ddl

    v_cramer = np.sqrt(
        chi2 / (n * min(lignes - 1, colonnes - 1))
    )

    return v_cramer, chi2, p_value, ddl


resultats = []

# 3. Étudier toutes les paires de variables qualitatives
for var1, var2 in combinations(variables_quali, 2):

    # Exclure les lignes où l'une des deux variables est manquante
    df_pair = X_quali[[var1, var2]].dropna()

    if df_pair.empty:
        continue

    tableau = pd.crosstab(df_pair[var1], df_pair[var2])

    # Une seule modalité : pas d'association calculable
    if tableau.shape[0] < 2 or tableau.shape[1] < 2:
        continue

    v_cramer, chi2, p_value, ddl = calculer_v_cramer(tableau)

    resultats.append({
        "variable_1": var1,
        "variable_2": var2,
        "chi2": chi2,
        "ddl": ddl,
        "p_value": p_value,
        "V_cramer": v_cramer,
        "n": tableau.to_numpy().sum()
    })

# 4. Résultats triés par intensité d'association
resultats_quali = pd.baseFrame(resultats)

if not resultats_quali.empty:
    resultats_quali = resultats_quali.sort_values(
        "V_cramer", ascending=False
    ).reset_index(drop=True)

print(resultats_quali)

## Liste des variables a probablement considérer pour l'analyse 

In [33]:

import polars as pl

TARGET = "ddefaut_ndb"
SEUIL_MANQUANTS = 0.95

# 1. IV > 0,5
vars_iv_suspect = (
    ivs.filter(pl.col("iv") > 0.5)
    .get_column("variable")
    .to_list()
)

# 2. IV < 0,02
vars_iv_faible = (
    ivs.filter(pl.col("iv") < 0.02)
    .get_column("variable")
    .to_list()
)

# 3. Variables strictement constantes
vars_constantes = (
    const.filter(pl.col("n_valeurs") == 1)
    .get_column("variable")
    .to_list()
)

# 4. Variables quasi constantes : modalité dominante >= 99 %
vars_quasi_constantes = (
    const.filter(
        (pl.col("part_top") >= 0.99)
        & (pl.col("n_valeurs") > 1)
    )
    .get_column("variable")
    .to_list()
)

# 5. Variables avec plus de 95 % de valeurs manquantes
# On compte les null et les NaN pour les colonnes numériques.
expressions_manquants = []

for col, dtype in zip(base.columns, base.dtypes):
    if col == TARGET:
        continue

    expr = pl.col(col).is_null()

    if dtype.is_numeric():
        expr = expr | pl.col(col).is_nan().fill_null(False)

    expressions_manquants.append(
        expr.mean().alias(col)
    )

taux_manquants = (
    base.select(expressions_manquants)
    .transpose(
        include_header=True,
        header_name="variable",
        column_names=["taux_manquants"]
    )
)

vars_trop_manquantes = (
    taux_manquants
    .filter(pl.col("taux_manquants") > SEUIL_MANQUANTS)
    .get_column("variable")
    .to_list()
)

# Union des variables à exclure
vars_a_exclure = set(
    vars_iv_suspect
    + vars_iv_faible
    + vars_constantes
    + vars_quasi_constantes
    + vars_trop_manquantes
)

# Ne jamais supprimer la cible
vars_a_exclure.discard(TARGET)

# Construire la base filtrée
colonnes_a_garder = [
    col for col in base.columns
    if col not in vars_a_exclure
]

base_filtre = base.select(colonnes_a_garder)

# Bilan
print("Base initiale :", base.shape)
print("Base filtrée :", base_filtre.shape)
print("Variables initiales :", len(base.columns))
print("Variables conservées :", len(colonnes_a_garder))
print("Variables exclues :", len(vars_a_exclure))
print("  IV > 0,5 :", len(vars_iv_suspect))
print("  IV < 0,02 :", len(vars_iv_faible))
print("  Constantes :", len(vars_constantes))
print("  Quasi constantes :", len(vars_quasi_constantes))
print("  Plus de 95 % de manquants :", len(vars_trop_manquantes))
print("Cible conservée :", TARGET in base_filtre.columns)

Base initiale : (78118, 173)
Base filtrée : (78118, 64)
Variables initiales : 173
Variables conservées : 64
Variables exclues : 109
  IV > 0,5 : 19
  IV < 0,02 : 80
  Constantes : 64
  Quasi constantes : 15
  Plus de 95 % de manquants : 12
Cible conservée : True


In [34]:
colonnes_a_garder = [
    col for col in base.columns
    if col not in vars_a_exclure
]

base_filtre = base.select(colonnes_a_garder)

print("Base initiale :", base.shape)
print("Base filtrée :", base_filtre.shape)
print("Cible conservée :", TARGET in base_filtre.columns)

Base initiale : (78118, 173)
Base filtrée : (78118, 64)
Cible conservée : True


In [36]:
import polars as pl

TARGET = "ddefaut_ndb"

# Vérifier les colonnes disponibles
print("Base filtrée :", base_filtre.shape)

bin_cols = [
    col for col in base_filtre.columns
    if col.endswith("_bin") and col != TARGET
]

print("Nombre de variables binées :", len(bin_cols))
print(bin_cols[:10])

Base filtrée : (78118, 64)
Nombre de variables binées : 0
[]


In [41]:

import polars as pl
import numpy as np
from sklearn.tree import DecisionTreeClassifier

def calculer_woe_iv_brut(
    df: pl.DataFrame,
    variables: list[str],
    target: str = "ddefaut_ndb",
    max_leaf_nodes: int = 5,
    min_samples_leaf: float = 0.02,
    lissage: float = 0.5,
):
    y = df[target].cast(pl.Int64).to_numpy()
    bons_total = int((y == 0).sum())
    defauts_total = int((y == 1).sum())

    if bons_total == 0 or defauts_total == 0:
        raise ValueError("La cible doit contenir des bons (0) et des défauts (1).")

    resultats = []
    mappings = {}

    for var in variables:
        s = df.get_column(var)
        valeurs = s.to_numpy()
        numerique = s.dtype.is_numeric()

        # Préparer les classes
        if numerique:
            x = s.cast(pl.Float64).to_numpy()
            valides = np.isfinite(x)

            classes = np.full(len(x), "__MANQUANT__", dtype=object)

            # Arbre appris uniquement sur les valeurs observées
            if valides.sum() > 0 and len(np.unique(y[valides])) == 2:
                arbre = DecisionTreeClassifier(
                    max_leaf_nodes=max_leaf_nodes,
                    min_samples_leaf=min_samples_leaf,
                    random_state=42,
                )
                arbre.fit(x[valides].reshape(-1, 1), y[valides])
                classes[valides] = arbre.apply(
                    x[valides].reshape(-1, 1)
                ).astype(str)
            else:
                classes[valides] = x[valides].astype(str)
        else:
            classes = np.array([
                "__MANQUANT__" if v is None else str(v)
                for v in valeurs
            ], dtype=object)

        # Regrouper les effectifs par classe
        tab = pl.DataFrame({
            "classe": classes.tolist(),
            "y": y,
        }).group_by("classe").agg(
            (pl.col("y") == 0).sum().alias("bons"),
            (pl.col("y") == 1).sum().alias("defauts"),
        )

        k = tab.height

        tab = tab.with_columns(
            ((pl.col("bons") + lissage) /
             (bons_total + lissage * k)).alias("part_bons"),
            ((pl.col("defauts") + lissage) /
             (defauts_total + lissage * k)).alias("part_defauts"),
        ).with_columns(
            (pl.col("part_bons") / pl.col("part_defauts"))
            .log().alias("woe")
        ).with_columns(
            (
                (pl.col("part_bons") - pl.col("part_defauts"))
                * pl.col("woe")
            ).alias("iv_classe")
        )

        iv = tab.get_column("iv_classe").sum()

        mappings[var] = {
            row["classe"]: row["woe"]
            for row in tab.select("classe", "woe").to_dicts()
        }

        resultats.append({
            "variable": var,
            "iv": float(iv),
            "n_classes": k,
            "n_bons": bons_total,
            "n_defauts": defauts_total,
        })

    if not resultats:
        return mappings, pl.DataFrame(
            schema={
                "variable": pl.String,
                "iv": pl.Float64,
                "n_classes": pl.Int64,
                "n_bons": pl.Int64,
                "n_defauts": pl.Int64,
            }
        )

    return mappings, pl.DataFrame(resultats).sort(
        "iv", descending=True
    )

In [43]:

TARGET = "ddefaut_ndb"

variables_filtrees = [
    c for c in base_filtre.columns
    if c not in [TARGET, "gdt", "datdelhis"]
]

woe_mappings_filtre, ivs_filtre = calculer_woe_iv_brut(
    base_filtre,
    variables_filtrees,
    target=TARGET,
)

print(ivs_filtre)

shape: (61, 5)
┌─────────────────────────────────┬──────────┬───────────┬────────┬───────────┐
│ variable                        ┆ iv       ┆ n_classes ┆ n_bons ┆ n_defauts │
│ ---                             ┆ ---      ┆ ---       ┆ ---    ┆ ---       │
│ str                             ┆ f64      ┆ i64       ┆ i64    ┆ i64       │
╞═════════════════════════════════╪══════════╪═══════════╪════════╪═══════════╡
│ SamplingWeight                  ┆ 5.913388 ┆ 2         ┆ 77669  ┆ 449       │
│ DATE_SAIN                       ┆ 1.059661 ┆ 133       ┆ 77669  ┆ 449       │
│ CAUSE_SAIN_M0                   ┆ 0.473091 ┆ 5         ┆ 77669  ┆ 449       │
│ NBJ_ARR_SS_max_Mm25_Mm36        ┆ 0.423546 ┆ 4         ┆ 77669  ┆ 449       │
│ CRTAD_IND_0015                  ┆ 0.385078 ┆ 4         ┆ 77669  ┆ 449       │
│ NBJ_ARR_SS_max_Mm19_Mm24        ┆ 0.383521 ┆ 3         ┆ 77669  ┆ 449       │
│ CRTAD_AG_NBJDE_BC               ┆ 0.379547 ┆ 5         ┆ 77669  ┆ 449       │
│ NBJDE_RATIO_A_vs_A1    

In [45]:
# ENregistrer cette base filtrée pour la suite du projet en parquet
base_filtre.write_parquet(
    "data/segments/mono_sans_engagement/base_mono_sans_engagement_filtre.parquet",
    compression="snappy"
)
